# v12

In [ ]:
# Step 1: DeepSeek credentials
import os, requests
api_key = os.environ['DEEPSEEK_API_KEY']
print(f'[ENV] KEY={api_key[:8]}...')
headers = {'Authorization': f'Bearer {api_key}', 'Content-Type': 'application/json'}
payload = {'model': 'deepseek-chat',
           'messages': [{'role': 'user', 'content': 'Say OK'}],
           'max_tokens': 10}
try:
    r = requests.post('https://api.deepseek.com/chat/completions',
                      headers=headers, json=payload, timeout=15)
    r.raise_for_status()
    print(f'[LLM PING] {r.json()["choices"][0]["message"]["content"]}')
except Exception as e:
    print(f'[LLM PING] ERROR: {e}')

In [ ]:
# Step 2: Load KG (full.txt) → fast dict index
import sys, os, time, json
from collections import defaultdict
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path: sys.path.insert(0, project_root)
from src.utils.data_structs import QuadrupletCreator, NodeCreator, RelationCreator, RelationType, NodeType
kg_path = os.path.join(project_root, 'wikidata_big/kg')

print('[MAP] Loading id->name mappings...')
ent_map, rel_map = {}, {}
with open(f'{kg_path}/wd_id2entity_text.txt') as f:
    for line in f:
        p = line.strip().split('\t')
        if len(p) >= 2: ent_map[p[0]] = p[1]
with open(f'{kg_path}/wd_id2relation_text.txt') as f:
    for line in f:
        p = line.strip().split('\t')
        if len(p) >= 2: rel_map[p[0]] = p[1]

# --- Load new entity/relation names from out.json ---
out_json_path = os.path.join(project_root, 'out.json')
out_data = []
if os.path.exists(out_json_path):
    try:
        with open(out_json_path, encoding='utf-8') as f:
            out_data = json.load(f)
        added_e, added_r = 0, 0
        for item in out_data:
            s = item.get('s', {})
            r = item.get('r', {})
            o = item.get('o', {})
            if s.get('id') and s.get('name') and s['id'] not in ent_map:
                ent_map[s['id']] = s['name']; added_e += 1
            if o.get('id') and o.get('name') and o['id'] not in ent_map:
                ent_map[o['id']] = o['name']; added_e += 1
            if r.get('id') and r.get('name') and r['id'] not in rel_map:
                rel_map[r['id']] = r['name']; added_r += 1
        print(f'[MAP] +{added_e} ent, +{added_r} rel from out.json')
    except Exception as e:
        print(f'[MAP] out.json load skipped: {e}')
else:
    print('[MAP] out.json not found, skipping')
print(f'[MAP] Total: {len(ent_map):,} entities, {len(rel_map):,} relations')

print('[GRAPH] Parsing full.txt...')
t0 = time.time()
quads_raw, errors, seen_ids = [], 0, set()
with open(f'{kg_path}/full.txt') as f:
    for line in f:
        p = line.strip().split('\t')
        if len(p) < 3: continue
        try:
            t_name = f'{p[3]} - {p[4]}' if len(p) > 4 else (p[3] if len(p) > 3 else 'Always')
            s_node = NodeCreator.create(NodeType.object, ent_map.get(p[0], p[0]), prop={'wd_id': p[0]})
            r_node = RelationCreator.create(RelationType.simple, name=rel_map.get(p[1], p[1]), prop={'wd_id': p[1]})
            o_node = NodeCreator.create(NodeType.object, ent_map.get(p[2], p[2]), prop={'wd_id': p[2]})
            t_node = NodeCreator.create(NodeType.time, t_name, add_stringified_node=True)
            quad = QuadrupletCreator.create(s_node, r_node, o_node, t_node)
            if quad.id not in seen_ids:
                quads_raw.append(quad); seen_ids.add(quad.id)
        except: errors += 1
print(f'[GRAPH] {len(quads_raw):,} quads in {time.time()-t0:.1f}s (err={errors})')

# --- Load facts (quadruplets) from out.json ---
if out_data:
    added_q, err_q = 0, 0
    for item in out_data:
        try:
            s = item.get('s', {})
            r = item.get('r', {})
            o = item.get('o', {})
            t_prop = item.get('t', {}).get('prop', {}) if isinstance(item.get('t'), dict) else {}
            t_start = str(t_prop.get('start', '')) if t_prop.get('start') is not None else ''
            t_end   = str(t_prop.get('end', ''))   if t_prop.get('end')   is not None else ''
            if t_start and t_end and t_start != t_end:
                t_name = f'{t_start} - {t_end}'
            elif t_start:
                t_name = t_start
            else:
                t_name = 'Always'
            s_node = NodeCreator.create(NodeType.object, ent_map.get(s.get('id', ''), s.get('name', '')), prop={'wd_id': s.get('id', '')})
            r_node = RelationCreator.create(RelationType.simple, name=rel_map.get(r.get('id', ''), r.get('name', '')), prop={'wd_id': r.get('id', '')})
            o_node = NodeCreator.create(NodeType.object, ent_map.get(o.get('id', ''), o.get('name', '')), prop={'wd_id': o.get('id', '')})
            t_node = NodeCreator.create(NodeType.time, t_name, add_stringified_node=True)
            quad = QuadrupletCreator.create(s_node, r_node, o_node, t_node)
            if quad.id not in seen_ids:
                quads_raw.append(quad); seen_ids.add(quad.id); added_q += 1
        except: err_q += 1
    print(f'[GRAPH] +{added_q} quads from out.json (err={err_q}, total: {len(quads_raw):,})')

print('[INDEX] Building...')
t1 = time.time()
wd_id_to_quads = defaultdict(list)
name_to_quads  = defaultdict(list)
for quad in quads_raw:
    s_wd = quad.start_node.prop.get('wd_id')
    o_wd = quad.end_node.prop.get('wd_id')
    if s_wd: wd_id_to_quads[s_wd].append(quad)
    if o_wd: wd_id_to_quads[o_wd].append(quad)
    name_to_quads[quad.start_node.name].append(quad)
    name_to_quads[quad.end_node.name].append(quad)
print(f'[INDEX] Done in {time.time()-t1:.1f}s')

In [ ]:
# Step 3: Init LLM + encoder + WikidataMapper
import torch
from sentence_transformers import SentenceTransformer
from src.llm.deepseek_client import DeepSeekClient
from src.utils.wikidata_utils import WikidataMapper

llm_client = DeepSeekClient(
    api_key=os.environ['DEEPSEEK_API_KEY'],
    model='deepseek-chat'
)
print('[LLM] DeepSeek ready')

FINETUNED_PATH = os.path.join(project_root, 'models/wikidata_finetuned_remote/wikidata_finetuned')
if os.path.exists(FINETUNED_PATH):
    encoder = SentenceTransformer(FINETUNED_PATH)
    print(f'[E5] encoder: finetuned ({FINETUNED_PATH})')
else:
    encoder = SentenceTransformer('intfloat/multilingual-e5-small')
    print('[E5] encoder: multilingual-e5-small (finetuned not found, fallback)')

mapper = WikidataMapper(kg_path)

# Sync mapper's internal dicts with dynamically loaded ent_map (e.g. from out.json)
synced = 0
for wd_id, name in ent_map.items():
    if wd_id not in mapper.id2name:
        mapper.id2name[wd_id] = name
        mapper.name2id[name.lower()] = wd_id
        synced += 1
print(f'[MAP] WikidataMapper ready (+{synced} new entries synced from ent_map)')

temporal_scorer = None
try:
    from src.kg_model.temporal.temporal_model import TemporalScorer
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    temporal_scorer = TemporalScorer(
        checkpoint_path="models/tcomplex_extended.ckpt",
        data_path="data/wikidata_extended/kg/tkbc_processed_data/wikidata_extended/",
        rank=200,  # checkpoint embeddings dim=400 → rank=200
        device=device
    )
    print(f'[TComplEx] Loaded (rank=200, tcomplex_extended.ckpt)')
except Exception as e:
    print(f'[TComplEx] Skipped: {e}')

print('\nAll systems ready!')

In [ ]:
# Step 4: Final QA Logic
import numpy as np
import time
import re
import json

# --- Helper Functions ---
def log_debug(title, content, show=False):
    """Prints a formatted debug block."""
    if not show: return
    try:
        if isinstance(content, str):
            c_str = content
        else:
            c_str = json.dumps(content, indent=2, ensure_ascii=False)
        print(f'\n[DBG] {title}\n{"-"*40}\n{c_str}\n{"-"*40}')
    except Exception:
        print(f'\n[DBG] {title}\n{"-"*40}\n{content}\n{"-"*40}')

def resolve_entity(name: str):
    """3-step entity resolution: exact match -> fuzzy KG search -> LLM normalization."""
    if not name: return None, None

    # Step 1: exact match
    wd_id = mapper.get_id(name)
    if wd_id: return name, wd_id

    # Step 2: fuzzy search in KG names, LLM picks best
    cands = mapper.search_names(name, limit=10)
    if not cands and ' ' in name:
        cands = mapper.search_names(name.split()[0], limit=10)
    if cands:
        if len(cands) == 1:
            return cands[0], mapper.get_id(cands[0])
        cand_str = ', '.join(repr(c) for c in cands)
        prompt = (f"A question mentions entity '{name}'. "
                  f"Which of these KG names best matches?\nOptions: {cand_str}\n"
                  f"Output ONLY the exact name from the list, nothing else.")
        try:
            choice = llm_client.generate(prompt).strip().strip("\"'")
            if choice in cands: return choice, mapper.get_id(choice)
        except Exception as e:
            print(f'[WARN] resolve_entity step 2 LLM failed: {e}')
        return cands[0], mapper.get_id(cands[0])

    # Step 3: LLM normalization (e.g. 'USA' -> 'United States of America')
    try:
        norm = llm_client.generate(
            f"What is the full official Wikidata/Wikipedia name of '{name}'? "
            f"Output ONLY the name, nothing else."
        ).strip().strip("\"'")
        wd_id = mapper.get_id(norm)
        if wd_id: return norm, wd_id
        cands2 = mapper.search_names(norm, limit=5)
        if cands2: return cands2[0], mapper.get_id(cands2[0])
    except Exception as e:
        print(f'[WARN] resolve_entity step 3 LLM failed: {e}')

    return name, None

def parse_years(quad):
    """Extracts (start_year, end_year) from a quad."""
    if not quad.time or quad.time.name == 'Always': return None, None
    m = re.findall(r'(\d{4})', quad.time.name)
    if not m: return None, None
    years = sorted([int(y) for y in m])
    return years[0], years[-1]

def build_anon_ctx(quads):
    """Build anonymized context using Wikidata Q/P IDs only."""
    lines, seen = [], set()
    for q in quads:
        s = q.start_node.prop.get('wd_id', '?')
        r = q.relation.prop.get('wd_id', '?')
        o = q.end_node.prop.get('wd_id', '?')
        t = q.time.name if q.time else 'Always'
        sig = f'{s}-{r}-{o}-{t}'
        if sig not in seen:
            seen.add(sig)
            lines.append(f'- {s} --[{r}]--> {o} (Time: {t})')
    return '\n'.join(lines)

def decode_ans(raw_ans):
    """Extracts Q-ID from LLM response."""
    m = re.search(r'([QP]\d+)', raw_ans)
    return m.group(1) if m else raw_ans

# --- System Prompt (fixed) ---
ANON_SYS = (
    "You are a pure logical reasoning engine. "
    "Answer ONLY based on the provided facts. "
    "FACT STRUCTURE: 'Subject_ID --[Relation_ID]--> Object_ID (Time: Range)'\n"
    "Rules:\n"
    "- For entity questions: output ONLY the Q-ID or P-ID (e.g. Q123).\n"
    "- For time/year questions: output ONLY the year or date range (e.g. 1925 or 1899 - 1917).\n"
    "- Do NOT use external knowledge.\n"
    "- If the answer is not in the facts: output NULL."
)

BEFORE_WORDS = ('before', 'prior to', 'earlier than', 'preceding', 'until', 'up to', 'by')
AFTER_WORDS  = ('after', 'since', 'following', 'post-', 'from', 'starting from', 'beyond')

# --- Main Functions ---

def map_id_to_name(qid):
    if qid.startswith('Q'):
        return f"{ent_map.get(qid, qid)} ({qid})"
    if qid.startswith('P'):
        return f"{rel_map.get(qid, qid)} ({qid})"
    return qid

# P3: Confidence-gap selection helper
def select_by_confidence_gap(results_sorted, min_f=2, max_f=7, gap=0.20):
    """Takes facts up to gap threshold, guaranteeing [min_f, max_f]."""
    if not results_sorted: return []
    top_conf = results_sorted[0]['conf']
    selected = []
    for r in results_sorted[:max_f]:
        # check BEFORE append: don't add noise after min_f
        if len(selected) >= min_f and (top_conf - r['conf']) > gap:
            break
        selected.append(r)
    return selected

def ask(question: str, anonymize: bool = True, debug: bool = False):
    SEP = '=' * 80
    print(f'\n{SEP}\nQUESTION: {question}\n{SEP}')
    t0 = time.time()
    timings = {}
    t_s = time.time()

    # 1. Extraction
    print('\n[1/4] Extracting...')
    ext = llm_client.extract_search_parameters(question)
    log_debug('EXTRACTION', ext, show=debug)
    timings['extract'] = round(time.time() - t_s, 2); t_s = time.time()

    entities     = ext.get('entities', [])
    query_time   = ext.get('time')
    anchor_ent   = ext.get('anchor_entity')
    anchor_event = ext.get('anchor_event')
    q_type       = ext.get('type', 'simple_entity')

    # 2. Config (P1: BASE_ALPHA only; alpha resolved after PASS 1; P4: search_k moved after unique)
    BASE_ALPHA = {'simple_time': 0.6, 'before_after': 0.45,
                  'time_join': 0.5, 'first_last': 0.5}.get(q_type, 0.3)
    filter_temporal = q_type in ('before_after', 'time_join')
    print(f'  Type={q_type} BASE_ALPHA={BASE_ALPHA}')

    # 3. HOP 1 (find anchor's attribute time via semantic ranking)
    resolved_time = query_time
    if anchor_ent and anchor_event and not query_time and q_type != 'simple_time':
        print(f'\n[2/4] HOP 1: Resolving time for "{anchor_event}" of "{anchor_ent}"...')
        rn, aw = resolve_entity(anchor_ent)
        if aw:
            all_hq = wd_id_to_quads.get(aw, [])
            if all_hq:
                ae_emb = encoder.encode(['query: ' + anchor_event])[0]
                hq_txts = [QuadrupletCreator.stringify(q)[1] for q in all_hq]
                hq_embs = encoder.encode(['passage: ' + t for t in hq_txts])
                hq_scored = sorted(
                    zip(all_hq, hq_embs),
                    key=lambda x: float(np.dot(ae_emb, x[1]) / (np.linalg.norm(ae_emb) * np.linalg.norm(x[1]) + 1e-9)),
                    reverse=True
                )
                hq = [q for q, _ in hq_scored[:15]]
            else:
                hq = []
            ctx = '\n'.join(QuadrupletCreator.stringify(q)[1] for q in hq)
            prompt = f"FACTS:\n{ctx}\n\nExtract ONLY the 4-digit YEAR for '{anchor_event}' of '{rn}':"
            try:
                raw = llm_client.generate(prompt).strip()
                m = re.search(r'(\d{4})', raw)
                if m:
                    resolved_time = m.group(1)
                    print(f'  -> Hop 1 resolved year: {resolved_time}')
            except Exception as e:
                print(f'[WARN] HOP1 LLM call failed: {e}')
    timings['hop1'] = round(time.time() - t_s, 2); t_s = time.time()

    # 4. Retrieval & Buckets
    search_time = resolved_time
    print(f'\n[3/4] Retrieving (time={search_time})...')
    candidates_raw = []
    resolved_entities = []
    for ent in entities:
        res_name, wd_id = resolve_entity(ent)
        resolved_entities.append((ent, res_name, wd_id))
        batch = wd_id_to_quads.get(wd_id, []) if wd_id else name_to_quads.get(ent, [])
        candidates_raw.extend(batch)
        print(f'  - {len(batch)} quads for "{res_name}"')

    # Pre-filter for before_after
    if q_type == 'before_after' and search_time:
        try:
            ref = int(search_time)
            is_before = any(w in question.lower() for w in BEFORE_WORDS)
            if not is_before and not any(w in question.lower() for w in AFTER_WORDS):
                print('[WARN] Cannot determine before/after direction from question text')
            filtered = [q for q in candidates_raw if (parse_years(q)[0] is not None and
                        (is_before and parse_years(q)[0] < ref or not is_before and parse_years(q)[1] > ref))]
            if filtered:
                candidates_raw = filtered
                print(f'  [B3] pre-filter applied: {len(filtered)} quads')
        except Exception as e:
            print(f'[WARN] before_after pre-filter failed: {e}')

    unique, seen = [], set()
    for q in candidates_raw:
        if q.id not in seen: unique.append(q); seen.add(q.id)
    if not unique:
        print('>>> NO CANDIDATES'); return 'Unknown'

    # P4: Adaptive search_k after unique is built
    n_unique = len(unique)
    search_k = min(n_unique, max(15, int(n_unique ** 0.55)))
    if q_type == 'time_join':
        search_k = max(search_k, 20)
    print(f'  [search_k] n_unique={n_unique} → search_k={search_k}')

    # Temporal bucket for time_join (reuse resolved_entities — no double resolve_entity call)
    temporal_bucket = []
    if q_type == 'time_join' and search_time:
        try:
            t_int = int(search_time)
            seen_b = set()
            for ent, res_name, wd_id in resolved_entities:
                all_q = wd_id_to_quads.get(wd_id, []) if wd_id else name_to_quads.get(ent, [])
                for q in all_q:
                    sy, ey = parse_years(q)
                    if sy is not None and sy <= t_int <= ey and q.id not in seen_b:
                        temporal_bucket.append(q); seen_b.add(q.id)
            print(f'  [B1/B2] temporal_bucket: {len(temporal_bucket)} quads @ {search_time}')
        except Exception as e:
            print(f'[WARN] temporal_bucket failed: {e}')

    # Scoring — PASS 1: collect E5 scores and TComplEx raw logits
    q_emb = encoder.encode(['query: ' + question])[0]
    txts = [QuadrupletCreator.stringify(q)[1] for q in unique]
    embs = encoder.encode(['passage: ' + t for t in txts])
    tcomplex_errors = 0
    raw_results = []
    for quad, text, emb in zip(unique, txts, embs):
        e5 = float(np.dot(q_emb, emb) / (np.linalg.norm(q_emb) * np.linalg.norm(emb) + 1e-9))
        tl = -100.0
        if search_time and temporal_scorer:
            sid = quad.start_node.prop.get('wd_id')
            rid = quad.relation.prop.get('wd_id')
            oid = quad.end_node.prop.get('wd_id')
            if sid and rid and oid:
                try:
                    tl = float(temporal_scorer.score(sid, rid, oid, str(search_time)))
                except Exception:
                    tcomplex_errors += 1
        raw_results.append({'text': text, 'quad': quad, 'e5': e5, 'tl': tl})
    if tcomplex_errors > 0:
        print(f'  [WARN] TComplEx failed on {tcomplex_errors}/{len(unique)} quads')

    # P1: Adaptive alpha — binary gate on max(valid_tls) and absolute count threshold
    MAX_LOGIT_THRESHOLD = -3.0  # TComplEx outputs -10.0 for unknown; -3.0 = lower bound for "known" entity
    MIN_TL_COUNT = 2             # need at least 2 candidates with real logits
    valid_tls = [r['tl'] for r in raw_results if r['tl'] > -99]
    if (not search_time or not temporal_scorer
            or len(valid_tls) < MIN_TL_COUNT
            or max(valid_tls) < MAX_LOGIT_THRESHOLD):
        alpha = 0.0  # pure E5
    else:
        alpha = BASE_ALPHA
    max_tl_str = f"{max(valid_tls):.2f}" if valid_tls else "N/A"
    print(f'  [alpha] valid_tls={len(valid_tls)} max_tl={max_tl_str} → alpha={alpha:.2f}')

    # PASS 2: min-max normalize tl → tp, compute final score
    if valid_tls and (max(valid_tls) - min(valid_tls)) > 1e-6:
        tl_min, tl_max = min(valid_tls), max(valid_tls)
        def tl_to_norm(tl): return (tl - tl_min) / (tl_max - tl_min) if tl > -99 else 0.5
    else:
        def tl_to_norm(tl): return 0.5
    results = []
    for r in raw_results:
        tp = tl_to_norm(r['tl'])
        final = (1 - alpha) * r['e5'] + alpha * tp if (search_time and temporal_scorer) else r['e5']
        # P2: temporal boost removed (pre-filter and temporal_bucket already handle relevance)
        results.append({'text': r['text'], 'quad': r['quad'], 'conf': final,
                        'e5': r['e5'], 'tp': tp, 'tl': r['tl']})
    results.sort(key=lambda x: x['conf'], reverse=True)
    top = results[:search_k]
    timings['score'] = round(time.time() - t_s, 2); t_s = time.time()

    log_debug('TOP RESULTS', '\n'.join(f"[{r['conf']:.3f}] E5={r['e5']:.3f} T={r['tp']:.3f} tl={r['tl']:.1f} | {r['text']}" for r in top[:10]), show=debug)

    # first_last sort
    if q_type == 'first_last':
        timed = [(parse_years(r['quad'])[0], r) for r in top if parse_years(r['quad'])[0] is not None]
        if timed:
            timed.sort(key=lambda x: x[0])
            isf = any(w in question.lower() for w in ('first', 'earliest', 'oldest', 'initial'))
            top = [c[1] for c in (timed[:5] if isf else timed[-5:])]

    # P3: Confidence-gap Selection (check-before-append)
    tb_set = set(id(q) for q in temporal_bucket)
    extra_results = [r for r in results if id(r['quad']) not in tb_set]
    n_max_extra = max(2, 7 - len(temporal_bucket))
    gap_selected = select_by_confidence_gap(extra_results, min_f=2, max_f=n_max_extra)
    selected_quads = temporal_bucket + [r['quad'] for r in gap_selected]
    print(f'  [select] tb={len(temporal_bucket)} + gap={len(gap_selected)} = {len(selected_quads)} facts')
    timings['select'] = round(time.time() - t_s, 2); t_s = time.time()

    # --- Step 6: Final Answer ---
    print('\n[4/4] Generating answer...')
    if anonymize:
        ctx = build_anon_ctx(selected_quads)
        anon_question = question
        for ent, res_name, wd_id in resolved_entities:
            if wd_id:
                anon_question = anon_question.replace(ent, wd_id)
                if res_name and res_name != ent:
                    anon_question = anon_question.replace(res_name, wd_id)
        answer_type = ext.get('answer_type', 'entity')
        if answer_type == 'year' or q_type == 'simple_time':
            answer_hint = 'ANSWER (year or date range only, e.g. "1925" or "1899 - 1917"):'
        else:
            answer_hint = 'ANSWER (Q-ID only, e.g. Q123):'
        um = f'QUESTION: {anon_question}\nTIME CONTEXT: {search_time}\nFACTS:\n{ctx}\n{answer_hint}'
        log_debug('PROMPT (ANON)', f'SYS: {ANON_SYS[:80]}...\nUSER: {um}', show=debug)
        try:
            raw = llm_client.generate(um, system=ANON_SYS).strip()
            if answer_type == 'year' or q_type == 'simple_time':
                if 'null' in raw.lower():
                    ans = 'Unknown'
                else:
                    m_range = re.search(r'(\d{4}\s*[-–]\s*\d{4})', raw)
                    m_year  = re.search(r'(\d{4})', raw)
                    ans = m_range.group(1) if m_range else (m_year.group(1) if m_year else raw)
            else:
                qid = decode_ans(raw)
                ans = map_id_to_name(qid)
        except Exception as e:
            ans = 'Error'
            print(f'[ERR] Generation failed: {e}')
    else:
        ctx = '\n'.join(f"- {r['text']}" for r in top)
        um = f'QUESTION: {question}\nFACTS:\n{ctx}\nANSWER:'
        try:
            ans = llm_client.generate(um, system="Answer concise Name only.").strip()
        except Exception as e:
            ans = 'Error'
            print(f'[ERR] Generation (non-anon) failed: {e}')

    timings['generate'] = round(time.time() - t_s, 2)
    timing_str = ' | '.join(f'{k}={v}s' for k, v in timings.items())
    print(f'\n{SEP}\n>>> FINAL ANSWER: {ans} ({time.time()-t0:.2f}s)\n[TIMING] {timing_str}\n{SEP}\n')
    return ans

def dbg(question, **kwargs):
    return ask(question, debug=True, **kwargs)

def ask_base(question: str, top_k: int = 10, anonymize: bool = True):
    """
    Pure Baseline (v2 alignment).
    - Strict Exact Match Entity Selection ONLY (no LLM fuzzy search).
    - Raw E5 ranking ONLY (no TComplEx, no temporal buckets).
    - Mandatory Anonymization (Q-ID mapping) to prevent LLM hallucination.
    """
    SEP = '=' * 80
    print(f'\n{SEP}\n[BASE] QUESTION: {question}\n{SEP}')
    t0 = time.time()

    # 1. Extraction (Using standard LLM extractor for fairness to get entities)
    ext = llm_client.extract_search_parameters(question)
    entities = ext.get('entities', [])
    print(f'  [1/4] Extracted Entities: {entities}')

    # 2. Strict V2 Retrieval (No fuzzy, no temporal buckets)
    craw = []
    for ent in entities:
        wid = mapper.get_id(ent)
        if wid:
            batch = wd_id_to_quads.get(wid, [])
            craw.extend(batch)
            print(f'  - Retrieved {len(batch)} quads for "{ent}" (Exact Match: {wid})')
        else:
            batch = name_to_quads.get(ent, [])
            craw.extend(batch)
            print(f'  - Retrieved {len(batch)} quads for "{ent}" (String Match)')

    unique, seen = [], set()
    for q in craw:
        if q.id not in seen: unique.append(q); seen.add(q.id)

    if not unique:
        print('>>> [BASE] NO CANDIDATES FOUND.'); return 'Unknown'

    # 3. Pure E5 Ranking
    print(f'  [2/4] Ranking {len(unique)} candidates using RAW E5...')
    qe = encoder.encode(['query: ' + question])[0]
    txts = [QuadrupletCreator.stringify(q)[1] for q in unique]
    embs = encoder.encode(['passage: ' + t for t in txts])

    res = []
    for q, t, e in zip(unique, txts, embs):
        sim = float(np.dot(qe, e) / (np.linalg.norm(qe) * np.linalg.norm(e) + 1e-9))
        res.append({'q': q, 't': t, 'c': sim})
    res.sort(key=lambda x: x['c'], reverse=True)
    top = res[:top_k]

    # 4. Final Inference with Minimal Prompt & Anonymization
    print('  [3/4] Formatting context...')
    if anonymize:
        ctx = build_anon_ctx([r['q'] for r in top])
        sys_msg = "Answer the question based ONLY on the provided facts. Output the corresponding ID (Q-id or P-id). If no answer is found, output NULL."
        um = f'QUESTION: {question}\nFACTS:\n{ctx}\nANSWER (ID only):'

        print('  [4/4] Generating answer...')
        try:
            raw_ans = llm_client.generate(um, system=sys_msg).strip()
            qid = decode_ans(raw_ans)
            ans = map_id_to_name(qid)
        except: ans = 'Error'
    else:
        ctx = '\n'.join(f"- {r['t']}" for r in top)
        sys_msg = "Answer based ONLY on the facts above. If unknown, say Unknown. Concise name only."
        um = f'QUESTION: {question}\nFACTS:\n{ctx}\nANSWER:'
        try: ans = llm_client.generate(um, system=sys_msg).strip()
        except: ans = 'Error'

    print(f'\n{SEP}\n>>> [BASE] FINAL ANSWER: {ans} ({time.time()-t0:.2f}s)\n{SEP}\n')
    return ans

print('ask(), ask_base() and dbg() ready.')

# QA Experiments v3 — Demo Questions
Running 7 questions to verify each type. All use `anonymize=True`.

In [ ]:
print('\n=== Q1: simple_entity  (in_kg=True) ===')
ask('Who was the spouse of Vladimir Nabokov?')

In [ ]:
print('\n=== Q2: simple_time  (in_kg=False — expects NULL/Unknown) ===')
ask('When was Vladimir Nabokov born?')

In [ ]:
print('\n=== Q3: before_after  (in_kg=True) ===')
ask('Where did Vladimir Nabokov live before 1930?')

In [ ]:
print('\n=== Q4: before_after  (in_kg=True) ===')
ask('Where did Vladimir Nabokov live after 1960?')

In [ ]:
print('\n=== Q5: first_last  (in_kg=False — expects NULL) ===')
ask('What was the first novel of Vladimir Nabokov?')

In [ ]:
print('\n=== Q6: time_join  (in_kg=True — key hallucination test) ===')
# v2 answer: 'Lyndon B. Johnson' from parametric memory (hallucination)
# v3 expected: Q9640=LBJ from KG via temporal_bucket (Q30 P6 Q9640 1963-1969)
ask('Who was the head of government of the United States when Nabokov was nominated for Nobel Prize in Literature?')

In [ ]:
print('\n=== Q7: relation  (in_kg=True) ===')
ask('What is the relation between Vladimir Nabokov and Vera Nabokova?')